# Filter Alerts by FIRMS Fire Detection

This notebook filters deforestation alerts by matching them with MODIS FIRMS (Fire Information for Resource Management System) fire detections.

## Purpose
- Load alert data from Google Earth Engine
- Match alerts with MODIS fire detections based on spatial and temporal proximity
- Return a filtered `ee.Image` that can be used with the Alert Navigator UI

## Output
The filtered image can be passed to `AlertGroupingNavigator` instead of an asset ID.

## 1. Import Libraries

In [10]:
import ee

# Initialize Google Earth Engine
try:
    ee.Initialize()
    print("✓ Google Earth Engine initialized successfully")
except Exception as e:
    print(f"✗ GEE initialization failed: {e}")
    print("Run 'earthengine authenticate' in terminal first")

✓ Google Earth Engine initialized successfully


## 2. Configuration Parameters

In [11]:
# ========== CONFIGURATION PARAMETERS ==========

# Alert asset
ALERT_ASSET_ID = "projects/ee-dfgm2006/assets/fires_colombia/Change_Alerts_S2"

# Forest mask asset (optional - uncomment to use)
FOREST_ASSET_ID = "projects/ee-dfgm2006/assets/fires_colombia/cambio2017-2018"

# Area of Interest (AOI)
AOI = ee.Geometry.Polygon(
        [[[-74.3110661666731, 1.7601569400641162],
          [-74.3110661666731, 1.1753294530083689],
          [-73.9869694869856, 1.1753294530083689],
          [-73.9869694869856, 1.7601569400641162]]]
    # [[[-74.19972795253469, 1.6116398432306993],
    #   [-74.19972795253469, 1.468182798154037],
    #   [-74.0078105331011, 1.468182798154037],
    #   [-74.0078105331011, 1.6116398432306993]]]
)




# FIRMS filtering parameters
DELTA_DAYS = 3  # Temporal window: ±3 days for matching alerts with fire detections
SCALE_FIRMS = 1000  # MODIS resolution in meters

print("Configuration set:")
print(f"  Alert Asset: {ALERT_ASSET_ID}")
print(f"  AOI defined")
print(f"  Temporal window: ±{DELTA_DAYS} days")

Configuration set:
  Alert Asset: projects/ee-dfgm2006/assets/fires_colombia/Change_Alerts_S2
  AOI defined
  Temporal window: ±3 days


## 3. Helper Functions

Date conversion functions for working with decimal year format.

In [12]:
def decimal_year_to_ee_date(decimal_year):
    """
    Convert decimal year to ee.Date.
    Example: 2024.5 -> July 2, 2024 (middle of the year)
    
    Args:
        decimal_year: ee.Number representing year as decimal
        
    Returns:
        ee.Date
    """
    dec = ee.Number(decimal_year)
    year = dec.floor()
    fraction = dec.subtract(year)
    
    start_of_year = ee.Date.fromYMD(year, 1, 1)
    next_year = start_of_year.advance(1, 'year')
    
    days_in_year = next_year.difference(start_of_year, 'day')
    days_offset = fraction.multiply(days_in_year)
    
    return start_of_year.advance(days_offset, 'day')


def date_to_decimal_year(date):
    """
    Convert ee.Date to decimal year.
    Example: July 2, 2024 -> 2024.5 (approximately)
    
    Args:
        date: ee.Date
        
    Returns:
        ee.Number representing decimal year
    """
    date = ee.Date(date)
    year = ee.Number.parse(date.format('Y'))
    start_of_year = ee.Date.fromYMD(year, 1, 1)
    next_year = start_of_year.advance(1, 'year')
    days_in_year = next_year.difference(start_of_year, 'day')
    days_since_start = date.difference(start_of_year, 'day')
    return year.add(days_since_start.divide(days_in_year))


print("✓ Helper functions defined")

✓ Helper functions defined


## 4. Load Alert Data and Determine Time Range

In [13]:
# Load alerts
alerts = ee.Image(ALERT_ASSET_ID)

# Optional: Apply forest mask
# forest = ee.Image(FOREST_ASSET_ID)
# fnf = forest.select("b1").eq(1).selfMask()
# alerts = alerts.updateMask(fnf)

print("Alert bands:", alerts.bandNames().getInfo())

# Get confirmation_date band
conf_date = alerts.select('confirmation_date')
conf_mask = conf_date.gt(0)

# Get time range from confirmation dates
conf_stats = conf_date.updateMask(conf_mask).reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=AOI,
    scale=500,  # Coarse scale for speed
    maxPixels=1e7
)

print("\nConfirmation date statistics (decimal years):")
print(conf_stats.getInfo())

# Extract min/max decimal years
min_dec = ee.Number(conf_stats.get('confirmation_date_min'))
max_dec = ee.Number(conf_stats.get('confirmation_date_max'))

# Convert to dates with buffer
start_date = decimal_year_to_ee_date(min_dec).advance(-7, 'day')
end_date = decimal_year_to_ee_date(max_dec).advance(7, 'day')

print(f"\nFIRMS time window:")
print(f"  Start: {start_date.format('YYYY-MM-dd').getInfo()}")
print(f"  End: {end_date.format('YYYY-MM-dd').getInfo()}")

Alert bands: ['last_stable_date', 'first_detection_date', 'confirmation_date', 'last_detection_date', 'confidence', 'difference', 'detection_count', 'monitoring_observation_count', 'calibration_observation_count']

Confirmation date statistics (decimal years):
{'confirmation_date_max': 2024.0809326171875, 'confirmation_date_min': 2023.9442138671875}

FIRMS time window:
  Start: 2023-12-04
  End: 2024-02-06


## 5. Prepare Confirmation Date at FIRMS Resolution

In [14]:
# Clip confirmation date to AOI
conf_date_1km = conf_date.clip(AOI)
conf_mask_1km = conf_mask.clip(AOI)

print(f"✓ Confirmation date prepared at {SCALE_FIRMS}m resolution")

✓ Confirmation date prepared at 1000m resolution


## 6. Build Fire-Alert Mask from MODIS FIRMS

Match MODIS fire detections with alert dates within the temporal window.

In [15]:
# Calculate temporal window in decimal years
delta_year = ee.Number(DELTA_DAYS).divide(365)

# Load MODIS FIRMS collection
modis_ic = ee.ImageCollection('FIRMS') \
    .filterBounds(AOI) \
    .filterDate(start_date, end_date)

print(f"Number of MODIS FIRMS images: {modis_ic.size().getInfo()}")

# Map over each FIRMS image to create fire-alert masks
def create_fire_alert_mask(img):
    """
    For each MODIS FIRMS image:
    1. Identify fire pixels (T21 > 0)
    2. Check temporal match with confirmation_date
    3. Return spatio-temporal match
    """
    # 1) Fire pixels: T21 > 0
    fire = img.select('T21').gt(0).selfMask().clip(AOI)
    
    # 2) Temporal match with confirmation_date
    img_date = ee.Date(img.get('system:time_start'))
    img_dec_year = date_to_decimal_year(img_date)
    
    dy_img = ee.Image.constant(img_dec_year).toFloat().clip(AOI)
    
    temporal_match = conf_date_1km \
        .subtract(dy_img) \
        .abs() \
        .lte(delta_year) \
        .updateMask(conf_mask_1km)
    
    # 3) Spatio-temporal match = fire AND temporal_match
    fire_alert = fire.And(temporal_match)
    
    return fire_alert.rename('fire_alert') \
        .copyProperties(img, ['system:time_start'])

fire_alert_ic = modis_ic.map(create_fire_alert_mask)

# Collapse over time (OR operation - max)
fire_alert_1km = fire_alert_ic.max().rename('fire_alert_1km').clip(AOI)

# Count fire alert pixels
fire_alert_count = fire_alert_1km.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=AOI,
    scale=1000,
    maxPixels=1e7
)

print(f"\nFire alert pixels detected: {fire_alert_count.getInfo()}")
print("✓ Fire-alert mask created")

Number of MODIS FIRMS images: 64

Fire alert pixels detected: {'fire_alert_1km': 0}
✓ Fire-alert mask created


## 7. Apply FIRMS Filter to Alerts

Create the final filtered alert image.

In [16]:
# Apply the fire-alert mask to the original alerts
alerts_filtered_by_firms = alerts.updateMask(fire_alert_1km)

print("✓ Alerts filtered by FIRMS")
print(f"Bands: {alerts_filtered_by_firms.bandNames().getInfo()}")

# Count filtered alert pixels
filtered_count = alerts_filtered_by_firms.select('difference').reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=AOI,
    scale=30,  # Native resolution
    maxPixels=1e7
)



✓ Alerts filtered by FIRMS
Bands: ['last_stable_date', 'first_detection_date', 'confirmation_date', 'last_detection_date', 'confidence', 'difference', 'detection_count', 'monitoring_observation_count', 'calibration_observation_count']


## Use with Alert Navigator UI

## 8. Export Filtered Alerts to Earth Engine Asset

In [8]:
# Export configuration
EXPORT_ASSET_ID = "projects/ee-dfgm2006/assets/fires_colombia/Change_Alerts_S2_filtered_by_FIRMS"
EXPORT_DESCRIPTION = "alerts_filtered_by_firms"

# Create export task
export_task = ee.batch.Export.image.toAsset(
    image=alerts_filtered_by_firms,
    description=EXPORT_DESCRIPTION,
    assetId=EXPORT_ASSET_ID,
    region=AOI,
    scale=30,  # Native resolution
    maxPixels=1e10,
    pyramidingPolicy={
        '.default': 'sample',
        'confirmation_date': 'sample',
        'difference': 'sample'
    }
)

# Start the export
export_task.start()

print("✓ Export task started!")
print(f"  Description: {EXPORT_DESCRIPTION}")
print(f"  Asset ID: {EXPORT_ASSET_ID}")
print(f"  Region: AOI")
print(f"  Scale: 30m")
print("\nMonitor the task at: https://code.earthengine.google.com/tasks")
print("Or check status with: ee.batch.Task.list()")


✓ Export task started!
  Description: alerts_filtered_by_firms
  Asset ID: projects/ee-dfgm2006/assets/fires_colombia/Change_Alerts_S2_filtered_by_FIRMS
  Region: AOI
  Scale: 30m

Monitor the task at: https://code.earthengine.google.com/tasks
Or check status with: ee.batch.Task.list()


In [21]:
# Uncomment to use the interactive UI with filtered alerts

from scripts.alert_grouping_navigator import AlertGroupingNavigator
from scripts.alert_navigator_ui import AlertNavigatorUI

# Create navigator with the filtered image
navigator = AlertGroupingNavigator(
    alert_image=alerts_filtered_by_firms,  # Use filtered image
    pixel_size=30
)

# Group alerts
alerts_gdf = navigator.group_alerts(
    aoi=ee.FeatureCollection([ee.Feature(AOI)]),
    min_alert_size_pixels=5,
    max_alerts=30,
    sorting_method="largest_first",
    grid_size=150000
)

# Display UI
ui = AlertNavigatorUI(navigator)
ui.display()

Detected CCDC-style alert format, converting to integer bands...


HTML(value='<h3>🛰️ Navigation & Imagery Controls</h3>')

HTML(value='<h3>📝 Alert Information</h3>')

Output()

HTML(value='<h3>🖼️ Satellite Imagery Visualization</h3>')

Output()


✓ Interactive navigator ready!
